# PVT v2 + MoE — supervised training & ablations (v10, Tutel backend)

Thin launcher over the **`pvt_moe`** package (all logic lives there — edit modules, not cells).
The **config cell** below is the only cell you should need to touch.

| # | Ablation | Config key | Example values |
|---|----------|-----------|----------------|
| 1 | Baseline PVT v2 | `model.ablation.use_moe` | `False` (dense) |
| 2 | MoE placement | `model.ablation.moe_placement` | `[[],[],[],[0,1]]` (stage 4), `[[],[],[],[1]]` (one block), `moe_last_n_stages: 2` (whole stages) |
| 3 | Norm | `model.norm_type` | `"layernorm"` / `"rmsnorm"` (fused; stage 4 keeps LN unless `stage4_keeps_layernorm: False`) |
| 4 | RoPE placement | `model.ablation.rope_placement` + `rope_theta` | same schema as MoE |
| 5 | Dataset | `dataset.name` | `"imagenet-1k"` / `"imagenet-22k"` (num_classes derived) |

MoE hyperparameters: `model.moe.{num_experts, top_k, capacity_factor, gate_noise}`.
Warm starts: `mode` = `"hf_pretrained"` (HF PVT v2 B1 + expert seeding) | `"scratch"` | `"ssl_init"` (JEPA backbone) | `"resume"`.

Before the first run: `python tests/run_all.py` from the repo root must pass.

In [ ]:
# Environment check — plain Jupyter on B200 (sm_100) or RTX 5090 (sm_120).
# Credentials: export HF_TOKEN / WANDB_API_KEY in the shell that starts Jupyter.
import importlib
import subprocess
import sys

import torch

print(f"torch {torch.__version__} | CUDA {torch.version.cuda}")
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability(0)
    print(f"GPU: {torch.cuda.get_device_name(0)} (sm_{cap[0]}{cap[1]})")
else:
    print("WARNING: no GPU visible — training will not be practical.")

_torch_minor = tuple(int(p) for p in torch.__version__.split("+")[0].split(".")[:2])
if _torch_minor < (2, 5):
    print("NOTE: torch >= 2.5 recommended (fast SDPA GQA path; >=2.4 for fused RMSNorm). "
          "The code falls back gracefully but slower.")

_required = ["pytorch_lightning", "torchmetrics", "timm", "datasets", "transformers",
             "huggingface_hub", "pandas", "matplotlib", "fvcore", "wandb"]
_missing = [m for m in _required if importlib.util.find_spec(m) is None]
if _missing:
    print(f"Installing: {_missing}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *_missing])

In [ ]:
# Tutel MoE backend (build from source once; takes a few minutes).
import importlib
import subprocess
import sys

if importlib.util.find_spec("tutel") is None:
    print("Building tutel from source ...")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-v", "-U", "--no-build-isolation",
        "git+https://github.com/microsoft/tutel@main",
    ])
import tutel

print("tutel OK:", tutel.__file__)

In [ ]:
# Make the repo importable (notebooks/ lives one level under the repo root).
import pathlib
import sys

cwd = pathlib.Path.cwd()
REPO_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
assert (REPO_ROOT / "pvt_moe").is_dir(), f"pvt_moe package not found under {REPO_ROOT}"
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from pvt_moe import default_config, merge_config, validate_config
from pvt_moe.data import build_dataloaders
from pvt_moe.engine import LitClassifier, build_ssl_trainer, build_trainer, setup_environment
from pvt_moe.models import build_model
from pvt_moe.utils import (
    count_flops,
    count_params,
    expert_utilization,
    plot_expert_utilization,
    plot_training_curves,
)

print(f"pvt_moe loaded from {REPO_ROOT}")

In [ ]:
# ============================== CONFIG =====================================
# The ONLY cell to edit. Defaults reproduce the v9 recipe:
# MoE (8 experts, top-1) + RoPE in both stage-4 blocks, LayerNorm, ImageNet-1k.
cfg = merge_config(default_config(), {
    "mode": "hf_pretrained",         # scratch | hf_pretrained | ssl_init | resume
    "ckpt_path": None,               # required for resume / ssl_init
    "epochs": 100,
    "batch_size": 1024,
    "num_workers": 12,
    "use_wandb": True,

    # ---- ablation axes (see table above) ----
    "model": {
        "norm_type": "layernorm",                    # or "rmsnorm"
        "ablation": {
            "use_moe": True,
            "moe_placement": [[], [], [], [0, 1]],   # stage-4, both blocks
            # "moe_last_n_stages": 2,                # convenience alternative
            "use_rope": True,
            "rope_placement": [[], [], [], [0, 1]],
            "rope_theta": 50.0,
        },
        "moe": {"backend": "tutel", "num_experts": 8, "top_k": 1,
                "capacity_factor": 2.0, "gate_noise": 0.5},
    },
    "dataset": {"name": "imagenet-1k"},
    "optim": {"lr": 1e-4, "warmup_epochs": 7, "stage4_lr_multiplier": 10.0},
})

cfg = validate_config(cfg)
print("run:", cfg["run_name"])

In [ ]:
device = setup_environment(cfg)

In [ ]:
train_loader, val_loader = build_dataloaders(cfg)
xb, yb = next(iter(train_loader))
print("batch:", xb.shape, yb.shape, xb.dtype)

In [ ]:
# Sanity gauntlet — run before EVERY training launch (seconds of insurance
# against hours of wasted GPU time).
import torch

# 1) Tiny CPU model, all ablations off: forward shape + no-aux contract.
_tiny = validate_config(merge_config(cfg, {
    "mode": "scratch",
    "model": {
        "embed_dims": [16, 32, 48, 64], "num_heads": [1, 2, 4, 4],
        "num_kv_heads": [1, 1, 2, 2], "mlp_ratios": [2, 2, 2, 2],
        "depths": [1, 1, 1, 1], "pretrained_hf_id": None,
        "ablation": {"use_moe": False, "moe_placement": [[], [], [], []],
                     "use_rope": False, "rope_placement": [[], [], [], []]},
    },
}))
_m = build_model(_tiny)
_logits, _aux = _m(torch.randn(2, 3, 224, 224))
assert _logits.shape == (2, cfg["dataset"]["num_classes"]), _logits.shape
assert _aux is None
print("tiny CPU forward OK")

# 2) Full architecture on GPU with the REAL ablation flags: aux must flow
#    through MoE blocks in train mode.
if torch.cuda.is_available():
    _probe = validate_config(merge_config(cfg, {"mode": "scratch",
                                                "model": {"pretrained_hf_id": None}}))
    _full = build_model(_probe).cuda().train()
    with torch.autocast("cuda", dtype=torch.bfloat16):
        _logits, _aux = _full(torch.randn(2, 3, 224, 224, device="cuda"))
    assert _logits.shape == (2, cfg["dataset"]["num_classes"])
    _abl = _probe["model"]["ablation"]
    if _abl["use_moe"] and any(_abl["moe_placement"]):
        assert _aux is not None and torch.isfinite(_aux), "MoE on but aux did not flow!"
        print(f"GPU aux-flow OK (aux={_aux.item():.4f})")
    _full.eval()
    del _full
    torch.cuda.empty_cache()
print("sanity checks passed")

In [ ]:
model = LitClassifier(cfg)
count_params(model.model)

In [ ]:
trainer = build_trainer(cfg)
resume_ckpt = cfg["ckpt_path"] if cfg["mode"] == "resume" else None
trainer.fit(model, train_loader, val_loader, ckpt_path=resume_ckpt)

In [ ]:
print("best:", trainer.checkpoint_callback.best_model_path)
print("last:", trainer.checkpoint_callback.last_model_path)

In [ ]:
import os

csv_dir = trainer.loggers[0].log_dir  # CSVLogger is always loggers[0]
plot_training_curves(os.path.join(csv_dir, "metrics.csv"),
                     save_path=os.path.join(csv_dir, "training_curves.png"))

In [ ]:
# Expert utilization — THE first diagnostic when an MoE run underperforms.
# Healthy top-1/8-expert routing: every expert 5-25%, entropy near 2.08.
if cfg["model"]["ablation"]["use_moe"]:
    # PL teardown moves the module to CPU after fit — put it back first.
    counts = expert_utilization(model.model.to(device), val_loader, num_batches=20)
    plot_expert_utilization(counts)

In [ ]:
count_flops(model.model, img_size=cfg["dataset"]["img_size"])

## Troubleshooting

| Symptom | Likely cause | Where to look |
|---|---|---|
| val acc ~0.1% from epoch 1 with `hf_pretrained` | weights not loading (remap drift) | the `[HF pretrained]` stats line — `loaded=` must be in the hundreds |
| `train_aux` stuck at 0 with MoE on | Tutel gates reverted to eval (gate_noise silently off) | `LitClassifier._force_tutel_gates_train` is load-bearing; check it runs |
| loss NaN spikes | aux-loss spike | clamp (`loss.aux_clamp`) + NaN guard already drop aux for the step; check `train_aux` curve |
| DataLoader workers die | workers x batch too big for RAM | lower `num_workers` / `batch_size` |
| one expert takes >50% tokens | routing collapse | expert-utilization cell above; consider higher `gate_noise` / `aux_weight` |

Full guidance: `.claude/skills/` (debug-training-run, run-ablation, moe-backends).